# Lab 1 - Tokenization Efficiency Benchmark

3 tokenizery, ten sam model nanoGPT.

In [1]:
import math
import platform
import re
import time
from collections import Counter

import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CPU:", platform.processor())
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "none")
print("PyTorch:", torch.__version__)

CPU: AMD64 Family 23 Model 96 Stepping 1, AuthenticAMD
GPU: NVIDIA GeForce GTX 1660 Ti
PyTorch: 2.6.0+cu124


## Dane

WikiText-103: trening - 450 000 linii `train`, test - cały `test` (1.28 MB).
Słowo = ciąg liter/cyfr albo znak interpunkcyjny.

In [2]:
wiki = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
train_texts = [t.strip() for t in wiki["train"][:450_000]["text"] if t.strip()]
test_texts = [t.strip() for t in wiki["test"]["text"] if t.strip()]


def split_words(text):
    return re.findall(r"\w+|[^\w\s]", text)


test_words = [w for text in test_texts for w in split_words(text)]
test_chars = sum(len(text) for text in test_texts)

print("train paragraphs:", len(train_texts))
print("test paragraphs:", len(test_texts))
print("test size [MB]:", sum(len(text.encode("utf-8")) for text in test_texts) / 1e6)
print("test words:", len(test_words))
print("test characters:", test_chars)

train paragraphs: 291502
test paragraphs: 2891
test size [MB]: 1.278983
test words: 249691
test characters: 1276949


## Tokenizer 1: GPT-2

Gotowy BPE z Hugging Face. Jego rozmiar słownika N = 50 257 używamy we wszystkich tokenizerach.

In [3]:
gpt2 = AutoTokenizer.from_pretrained("gpt2")
N = len(gpt2)

print("vocab size:", N)
print(gpt2.tokenize("Tokenization is the first step of every language model."))

vocab size: 50257
['Token', 'ization', 'Ġis', 'Ġthe', 'Ġfirst', 'Ġstep', 'Ġof', 'Ġevery', 'Ġlanguage', 'Ġmodel', '.']


## Tokenizer 2: whitespace

Słownik: `<UNK>` + N-1 najczęstszych słów.

In [4]:
class WhitespaceTokenizer:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.vocab = []
        self.word_to_id = {}

    def train(self, texts):
        counts = Counter()
        for text in texts:
            counts.update(split_words(text))
        most_common = [word for word, count in counts.most_common(self.vocab_size - 1)]
        self.vocab = ["<UNK>"] + most_common
        self.word_to_id = {word: i for i, word in enumerate(self.vocab)}

    def tokenize(self, text):
        return [word if word in self.word_to_id else "<UNK>" for word in split_words(text)]

    def encode(self, text):
        return [self.word_to_id[token] for token in self.tokenize(text)]

In [5]:
start = time.time()
ws = WhitespaceTokenizer(N)
ws.train(train_texts)
ws_train_time = time.time() - start

print("vocab size:", len(ws.vocab))
print("training time [s]:", ws_train_time)
print(ws.tokenize("Tokenization is the first step of every language model."))

vocab size: 50257
training time [s]: 12.098878145217896
['<UNK>', 'is', 'the', 'first', 'step', 'of', 'every', 'language', 'model', '.']


## Tokenizer 3: SentencePiece (BPE)

Słownik: N.

In [6]:
start = time.time()
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(train_texts),
    model_prefix="spm_wiki",
    vocab_size=N,
    model_type="bpe",
    max_sentence_length=100_000,
    minloglevel=2,
)
sp = spm.SentencePieceProcessor(model_file="spm_wiki.model")
sp_train_time = time.time() - start

print("vocab size:", sp.get_piece_size())
print("training time [s]:", sp_train_time)
print(sp.encode("Tokenization is the first step of every language model.", out_type=str))

vocab size: 50257
training time [s]: 10.99104118347168
['▁Tok', 'en', 'ization', '▁is', '▁the', '▁first', '▁step', '▁of', '▁every', '▁language', '▁model', '.']


In [7]:
tokenizers = {
    "gpt2": gpt2.tokenize,
    "whitespace": ws.tokenize,
    "sentencepiece": lambda text: sp.encode(text, out_type=str),
}

encoders = {
    "gpt2": gpt2.encode,
    "whitespace": ws.encode,
    "sentencepiece": sp.encode,
}

## Model (nanoGPT)

Taki sam dla wszystkich tokenizerów: 4 warstwy, 4 głowy, embedding 256, kontekst 256, batch 16, AdamW lr 5e-4, 1 epoka, seed 0.

In [8]:
CONTEXT = 256
EMBED = 256
LAYERS = 4
HEADS = 4
BATCH_SIZE = 16
LR = 5e-4


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(EMBED)
        self.attn = nn.MultiheadAttention(EMBED, HEADS, batch_first=True)
        self.ln2 = nn.LayerNorm(EMBED)
        self.mlp = nn.Sequential(nn.Linear(EMBED, 4 * EMBED), nn.GELU(), nn.Linear(4 * EMBED, EMBED))

    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask)[0]
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(N, EMBED)
        self.pos_emb = nn.Embedding(CONTEXT, EMBED)
        self.blocks = nn.Sequential(*[Block() for _ in range(LAYERS)])
        self.ln = nn.LayerNorm(EMBED)
        self.head = nn.Linear(EMBED, N, bias=False)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device)
        h = self.token_emb(x) + self.pos_emb(positions)
        h = self.blocks(h)
        return self.head(self.ln(h))


print("parameters:", sum(p.numel() for p in GPT().parameters()))

parameters: 28956672


### Trening

Tekst sklejony i pocięty na sekwencje po 257 tokenów.

In [9]:
def train(encode):
    ids = [i for text in train_texts for i in encode(text)]
    n = len(ids) // (CONTEXT + 1)
    sequences = torch.tensor(ids[: n * (CONTEXT + 1)]).view(n, CONTEXT + 1)

    torch.manual_seed(0)
    sequences = sequences[torch.randperm(n)]
    batches = sequences.split(BATCH_SIZE)
    model = GPT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    model.train()
    start = time.time()
    for step, batch in enumerate(batches):
        batch = batch.to(device)
        x, y = batch[:, :-1], batch[:, 1:]
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, N), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 500 == 0:
            print(f"step {step}/{len(batches)}, loss {loss.item():.2f}")
    return model, time.time() - start

In [10]:
models = {}
train_times = {}

for name, encode in encoders.items():
    print("===", name)
    models[name], train_times[name] = train(encode)
    print(f"training time: {train_times[name] / 60:.1f} min")

=== gpt2


step 0/7012, loss 10.99


step 500/7012, loss 6.47


step 1000/7012, loss 5.98


step 1500/7012, loss 5.88


step 2000/7012, loss 5.63


... (36 outputs left out)

step 5500/6591, loss 5.07


step 6000/6591, loss 5.05


step 6500/6591, loss 5.07


training time: 19.0 min


## Ewaluacja

word PPL = exp(NLL / liczba słów), char PPL = exp(NLL / liczba znaków).

In [ ]:
@torch.no_grad()
def evaluate(model, encode):
    ids = torch.tensor([i for text in test_texts for i in encode(text)])
    model.eval()
    total_nll = 0.0
    start = time.time()
    for i in range(0, len(ids) - 1, CONTEXT):
        x = ids[i : i + CONTEXT]
        y = ids[i + 1 : i + CONTEXT + 1]
         
        x = x[: len(y)]
        logits = model(x[None].to(device))[0]
        total_nll += F.cross_entropy(logits, y.to(device), reduction="sum").item()
    return total_nll, len(ids), time.time() - start

In [12]:
results = {}

for name, encode in encoders.items():
    nll, n_tokens, eval_time = evaluate(models[name], encode)
    results[name] = {
        "word PPL": math.exp(nll / len(test_words)),
        "char PPL": math.exp(nll / test_chars),
        "training time [min]": train_times[name] / 60,
        "inference time [s]": eval_time,
        "inference [tokens/s]": n_tokens / eval_time,
    }

pd.DataFrame(results).T.round(3)

,word PPL,char PPL,training time [min],inference time [s],inference [tokens/s]
gpt2,231.930,2.901,20.062,8.791,31605.488
whitespace,98.630,2.454,17.951,7.474,33409.144
sentencepiece,210.837,2.847,18.993,7.711,33931.637


## OOV (whitespace)

In [13]:
oov_words = [w for w in test_words if w not in ws.word_to_id]

print("OOV words:", len(oov_words), "/", len(test_words))
print(f"OOV words [%]: {100 * len(oov_words) / len(test_words):.2f}")
print("unique OOV words:", len(set(oov_words)), "/", len(set(test_words)))
print(f"unique OOV words [%]: {100 * len(set(oov_words)) / len(set(test_words)):.2f}")
print("most frequent OOV words:", Counter(oov_words).most_common(10))

OOV words: 8709 / 249691
OOV words [%]: 3.49
unique OOV words: 4513 / 21029
unique OOV words [%]: 21.46
most frequent OOV words: [('Bintulu', 169), ('temnospondyls', 108), ('Olmec', 55), ('Tetrarchs', 48), ('Rifenburg', 42), ('Tetrarch', 41), ('Yongsan', 39), ('Intramuros', 35), ('Gazan', 34), ('Mortier', 33)]


## Statystyki tokenizacji

Słowo zakodowane bezpośrednio = dokładnie 1 token (nie `<UNK>`).

In [14]:
def is_direct(name, word):
    if name == "gpt2":
        return len(gpt2.tokenize(" " + word)) == 1
    if name == "whitespace":
        return word in ws.word_to_id
    if name == "sentencepiece":
        ids = sp.encode(word)
        return len(ids) == 1 and ids[0] != sp.unk_id()

In [15]:
word_counts = Counter(test_words)
stats = {}

for name, encode in encoders.items():
    start = time.time()
    n_tokens = sum(len(encode(text)) for text in test_texts)
    encode_time = time.time() - start

    direct_words = [w for w in word_counts if is_direct(name, w)]
    stats[name] = {
        "tokens": n_tokens,
        "tokens/word": n_tokens / len(test_words),
        "directly encoded words [%]": 100 * sum(word_counts[w] for w in direct_words) / len(test_words),
        "unique words in dictionary": len(direct_words),
        "unique words in dictionary [%]": 100 * len(direct_words) / len(word_counts),
        "throughput [tokens/s]": n_tokens / encode_time,
    }

pd.DataFrame(stats).T.round(3)

,tokens,tokens/word,directly encoded words [%],unique words in dictionary,unique words in dictionary [%],throughput [tokens/s]
gpt2,277853.0,1.113,92.351,13018.0,61.905,322065.005
whitespace,249691.0,1.000,96.512,16516.0,78.539,1568726.077
sentencepiece,261647.0,1.048,95.046,14823.0,70.488,542744.158


## Analiza jakościowa

`Ġ` i `▁` = spacja przed tokenem.

In [16]:
samples = [
    "The city council approved a new plan to build a public library near the river. The building will open "
    "next year and will include a large reading room, a small cafe, and a garden where children can play.",
    "Mitochondrial dysfunction has been implicated in neurodegenerative disorders such as amyotrophic lateral "
    "sclerosis. Researchers used electroencephalography and immunohistochemistry to characterise the "
    "phosphorylation of tau proteins in transgenic mice over several months.",
    "On 14 March 2023 Grzegorz Brzęczyszczykiewicz from Łódź posted: \"Zażółć gęślą jaźń!\" The price of the "
    "new GPU rose to $1,299.99, and fans replied with #unbelievable while others compared it to the RTX4090.",
]

rows = []

for k, text in enumerate(samples, 1):
    words = split_words(text)
    print(f"=== sample {k} ({len(words)} words)")
    print(text)
    for name, tokenize in tokenizers.items():
        tokens = tokenize(text)
        print(f"\n[{name}] {len(tokens)} tokens")
        print(" | ".join(tokens))
        rows.append({
            "sample": k,
            "tokenizer": name,
            "words": len(words),
            "tokens": len(tokens),
            "tokens/word": len(tokens) / len(words),
            "directly encoded [%]": 100 * sum(is_direct(name, w) for w in words) / len(words),
        })
    print()

=== sample 1 (42 words)
The city council approved a new plan to build a public library near the river. The building will open next year and will include a large reading room, a small cafe, and a garden where children can play.

[gpt2] 42 tokens
The | Ġcity | Ġcouncil | Ġapproved | Ġa | Ġnew | Ġplan | Ġto | Ġbuild | Ġa | Ġpublic | Ġlibrary | Ġnear | Ġthe | Ġriver | . | ĠThe | Ġbuilding | Ġwill | Ġopen | Ġnext | Ġyear | Ġand | Ġwill | Ġinclude | Ġa | Ġlarge | Ġreading | Ġroom | , | Ġa | Ġsmall | Ġcafe | , | Ġand | Ġa | Ġgarden | Ġwhere | Ġchildren | Ġcan | Ġplay | .

[whitespace] 42 tokens
The | city | council | approved | a | new | plan | to | build | a | public | library | near | the | river | . | The | building | will | open | next | year | and | will | include | a | large | reading | room | , | a | small | cafe | , | and | a | garden | where | children | can | play | .

[sentencepiece] 42 tokens
▁The | ▁city | ▁council | ▁approved | ▁a | ▁new | ▁plan | ▁to | ▁build | ▁a | ▁public | ▁

In [17]:
pd.DataFrame(rows).set_index(["sample", "tokenizer"]).round(2)

words  tokens  tokens/word  directly encoded [%]
sample tokenizer                                                      
1      gpt2              42      42         1.00                100.00
       whitespace        42      42         1.00                100.00
       sentencepiece     42      42         1.00                100.00
2      gpt2              33      51         1.55                 72.73
       whitespace        33      33         1.00                 78.79
       sentencepiece     33      54         1.64                 69.70
3      gpt2              45      85         1.89                 82.22
       whitespace        45      45         1.00                 82.22
       sentencepiece     45      77         1.71                 80.00